In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42

# Load & clean data
df = pd.read_csv("../data/heart.csv")
df = df.replace("?", np.nan)

for col in ["ca", "thal"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop completely empty column if present
if "thal" in df.columns and df["thal"].isnull().all():
    df = df.drop(columns=["thal"])

y = (df["num"] > 0).astype(int)
X = df.drop(columns=["num"]).select_dtypes(exclude="object")

# Model pipeline
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        C=1.0, solver="liblinear", max_iter=1000, random_state=RANDOM_STATE
    ))
])

# Stratified CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_scores = cross_val_score(model, X, y, cv=skf, scoring="accuracy")

print("Fold accuracies:", cv_scores)
print("Mean accuracy:", cv_scores.mean())
print("Std deviation:", cv_scores.std())


Fold accuracies: [0.80978261 0.80978261 0.82065217 0.72826087 0.78804348]
Mean accuracy: 0.7913043478260869
Std deviation: 0.0332544750886486


In [2]:
from sklearn.model_selection import GridSearchCV, cross_val_score

# Inner CV (model selection)
param_grid = {
    "logreg__C": [0.1, 1, 10]
}

inner_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        solver="liblinear", max_iter=1000, random_state=RANDOM_STATE
    ))
])

inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    inner_model,
    param_grid,
    cv=inner_cv,
    scoring="accuracy"
)

# Outer CV (generalization estimate)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

nested_scores = cross_val_score(
    grid, X, y, cv=outer_cv, scoring="accuracy"
)

print("Nested CV scores:", nested_scores)
print("Nested CV mean accuracy:", nested_scores.mean())
print("Nested CV std:", nested_scores.std())


Nested CV scores: [0.81521739 0.80434783 0.82065217 0.72826087 0.78804348]
Nested CV mean accuracy: 0.7913043478260869
Nested CV std: 0.03343164456354281
